<a href="https://colab.research.google.com/github/hannahvadakken/CS439/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installs packages:

In [ ]:
!pip install -q peft transformers datasets

# Default installed version of torchao conflicts with peft
!pip uninstall -y torchao peft accelerate transformer-engine
!rm -rf /usr/local/lib/python3.12/dist-packages/torchao

!pip install --upgrade peft accelerate transformers

import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from datasets import load_dataset
import torchaudio
from torch.utils.data import Dataset
from transformers import ASTConfig, ASTModel
import matplotlib.pyplot as plt
from peft import LoraConfig, get_peft_model


print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Active: {torch.cuda.is_available()}")

Downloads the dataset:

In [ ]:
!mkdir -p /content/dataset

!wget -O /content/dataset/fan.zip https://zenodo.org/record/6529888/files/fan.zip?download=1
!unzip -q /content/dataset/fan.zip -d /content/dataset/
!rm /content/dataset/fan.zip
print("Download complete!")

Transforming audio to spectrograms

In [ ]:
from transformers import ASTFeatureExtractor

class ASTAnomalyDataset(torch.utils.data.Dataset):
    def __init__(self, file_list):
        self.file_list = file_list
        # This guarantees the audio is processed exactly how the model was trained
        self.feature_extractor = ASTFeatureExtractor(
            sampling_rate=16000,    # MIMII is usually 16kHz
            num_mel_bins=128,       # AST expects 128 frequency bins
            max_length=1024,        # This equals 10.24 seconds (MIMII clips are 10s)
            do_normalize=True,      # Crucial for MSELoss
            return_tensors="pt"     # Returns PyTorch tensors directly

        )

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        # Load audio
        waveform, sr = torchaudio.load(self.file_list[idx])

        # Ensure mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # The extractor handles: Spectrogram + Padding/Trimming to 1024 + Normalization
        inputs = self.feature_extractor(
            waveform.squeeze().numpy(),
            sampling_rate=sr,
            return_tensors="pt"
        )

        label = 1 if "anomaly" in self.file_list[idx] else 0

        # Output shape: [1024, 128]
        return (inputs['input_values'].squeeze(0), label)

Sets up the initial AST model:

In [ ]:
#model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
configuration = ASTConfig()

# Initializing a model (with random weights) from the MIT/ast-finetuned-audioset-10-10-0.4593 style configuration
model = ASTModel(configuration)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")

LoRA Config

In [ ]:
config = LoraConfig(
    r=8,               # The 'Rank' (How much new 'memory' we give the model)
    lora_alpha=32,     # Scaling factor (higher = more influence for the new weights)
    target_modules=["query", "value"], # The specific Attention layers to adapt
    lora_dropout=0.1,  # Helps prevent the model from just memorizing the training set
    bias="none"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()


Creating the Auto-Encoder:

In [ ]:
class YourASTAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. The AST acts as the "Encoder"
        self.encoder = model

        # 2. Re-shaping the decoder to match your frequency bins (128)
        self.decoder = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Linear(512, 128) # Map back to the 128 mel bins
        )

    def forward(self, x):
        # x shape: [batch, 1024, 128]

        # HuggingFace AST returns an object, we need the 'last_hidden_state'
        # Shape: [batch, 1214, 768]
        enc_output = self.encoder(x).last_hidden_state

        # The decoder will automatically apply to the last dimension (768 -> 128)
        # Shape: [batch, 1214, 128]
        reconstructed = self.decoder(enc_output)

        # Final Step: Slice it to match your original time length (1024)
        # We ignore the extra 'summary' tokens AST adds at the start
        return reconstructed[:, :1024, :]

Initialize the data and some basic parameters

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import glob

# 1. Get the list of files
train_files = glob.glob("/content/dataset/fan/train/*.wav")

# 2. Initialize Dataset
# Make sure your ASTAnomalyDataset is designed to take a LIST of strings
train_dataset = ASTAnomalyDataset(train_files)

# 3. Hyperparameters
# Lowering Learning Rate: 0.01 is way too high for AST; 1e-4 is safer.
learning_rate = 1e-4
batch_size = 8
epochs = 10

autoencoder = YourASTAutoencoder()
autoencoder.to(device)
optimizer = optim.Adam(autoencoder.parameters(), lr=0.0001)
loss_fn = nn.MSELoss()

# 4. DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

print(f"Total training samples: {len(train_dataset)}")

The Training Loop:

In [ ]:
# --- TRAINING LOOP ---
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for i, batch in enumerate(train_loader):

        # 1. Load data
        # Depending on your dataset class, batch might be [tensor] or just tensor
        if isinstance(batch, (list, tuple)):
            inputs = batch[0].to(device)
        else:
            inputs = batch.to(device)

        # 2. Forward Pass
        # The model takes a spectrogram and outputs a reconstructed version
        outputs = autoencoder(inputs)

        #print(inputs.shape)
        #print(outputs.shape)

        # 3. Calculate Loss (Reconstruction Error)
        # We compare the output directly to the original input
        loss = loss_fn(outputs, inputs)

        # 4. Backward Pass (The "Learning" part)
        optimizer.zero_grad() # Clear previous gradients
        loss.backward()      # Compute new gradients
        optimizer.step()      # Update model weights

        running_loss += loss.item()

        # Print progress every few batches so you know it hasn't frozen
        if(i % 10 == 0):
          print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.6f}")

    # Average loss for the entire epoch
    avg_loss = running_loss / len(train_loader)
    print(f"--- End of Epoch {epoch+1} | Average Loss: {avg_loss:.6f} ---")

print("Training finished!")

Testing:

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import torch

def evaluate_model(model, loader, threshold, device):
    model.eval()

    y_true = []
    y_pred = []
    scores = []

    anom_sum = 0
    anom_count = 0
    normal_sum = 0
    normal_count = 0

    print(f"Starting Evaluation (Threshold: {threshold})...")

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)

            # 1. Model creates reconstruction (Model is BLIND to the label)
            reconstruction = model(inputs)

            # 2. Calculate the "Suspicion Score" (MSE)
            # We compute MSE between the input and the model's attempt to redraw it
            loss = loss_fn(reconstruction, inputs).item()


            if(labels.item() == 1):
              anom_sum += loss
              anom_count += 1
            else:
              normal_sum += loss
              normal_count += 1


            # 3. Apply Decision Rule
            prediction = 1 if loss >= threshold else 0

            # 4. Store results for the Final Report
            y_true.append(labels.item()) # The "Answer Key"
            y_pred.append(prediction)    # The Model's "Guess"
            scores.append(loss)

    # --- FINAL METRICS ---
    print("\n" + "="*30)
    print("EVALUATION COMPLETE")
    print("="*30)

    # Classification Report gives you Accuracy, Precision, and Recall
    print(classification_report(y_true, y_pred, target_names=['Working', 'Malfunctioning']))

    # Confusion Matrix shows specifically where the model was "fooled"
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print(f"True Negatives (Correctly Not Flagged): {tn}")
    print(f"False Positives (Incorrectly Flagged):  {fp}")
    print(f"False Negatives (Incorrectly Not Flagged): {fn}")
    print(f"True Positives (Correctly Flagged): {tp}")

    print("normal:" + str(normal_sum/normal_count))
    print("anom:" + str(anom_sum/anom_count))

    return y_true, y_pred, scores

# Example Execution:
# test_loader = DataLoader(FanDataset(test_file_paths), batch_size=1)


In [ ]:
test_files = glob.glob("/content/dataset/fan/test/*.wav")

test_loader = DataLoader(ASTAnomalyDataset(test_files), batch_size=1)
results = evaluate_model(autoencoder, test_loader, threshold=0.0195, device=device)